# 06 - Aplicar normalización final de autores y afiliaciones

Este notebook aplica las decisiones contextuales V2.3 sobre `autores_unam_completos.csv`.

Solo modifica `Autor_norm`, `Afiliacion1` y `Afiliacion2`. El resto de las columnas se conserva exactamente.

In [5]:
import pandas as pd

carpeta = "../04_Limpieza/02_normalizacion"

archivo_entrada = f"{carpeta}/autores_unam_completos.csv"
archivo_auditoria = f"{carpeta}/auditoria_normalizacion_contextual_v2_3.csv"
archivo_dic_autores = f"{carpeta}/diccionario_autores_unam_v2_3.csv"
archivo_dic_afiliaciones = f"{carpeta}/diccionario_afiliaciones_unam_v2_3.csv"

archivo_salida = f"{carpeta}/autores_unam_normalizados.csv"

columnas = [
    "Base_origen", "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_norm", "Afiliacion1", "Afiliacion2", "ISBN", "ISSN",
    "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"
]


In [6]:
# Leer todo como texto para no alterar Año, ISBN, ISSN, DOI u otros campos
entrada = pd.read_csv(
    archivo_entrada,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

auditoria = pd.read_csv(
    archivo_auditoria,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

dic_autores = pd.read_csv(
    archivo_dic_autores,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

dic_afiliaciones = pd.read_csv(
    archivo_dic_afiliaciones,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

# Trabajar solamente con las 16 columnas canónicas
faltantes = [c for c in columnas if c not in entrada.columns]
if faltantes:
    raise ValueError(f"Faltan columnas en autores_unam_completos.csv: {faltantes}")

entrada = entrada[columnas].copy()

print("Filas de entrada:", len(entrada))
print("Filas de auditoría:", len(auditoria))
print("Autores del diccionario:", len(dic_autores))
print("Afiliaciones del diccionario:", len(dic_afiliaciones))


FileNotFoundError: [Errno 2] No such file or directory: '../04_Limpieza/02_normalizacion/autores_unam_completos.csv'

In [ ]:
# Validaciones antes de aplicar cambios
if len(entrada) != len(auditoria):
    raise ValueError("La entrada y la auditoría contextual no tienen el mismo número de filas.")

if len(entrada) != 4266:
    print(f"Aviso: se esperaban 4266 filas y se encontraron {len(entrada)}.")

# La auditoría debe estar completamente resuelta a nivel contextual
if not auditoria["Estado_autor"].eq("RESUELTO").all():
    raise ValueError("Todavía existen autores no resueltos en la auditoría contextual.")

if not auditoria["Estado_afiliacion"].eq("RESUELTO").all():
    raise ValueError("Todavía existen afiliaciones no resueltas en la auditoría contextual.")

# Los diccionarios globales pueden contener CONFLICTO porque esos casos
# se resolvieron por publicación en la auditoría contextual, pero no REVISAR.
if dic_autores["Estado"].eq("REVISAR").any():
    raise ValueError("El diccionario de autores todavía contiene casos REVISAR.")

if dic_afiliaciones["Estado"].eq("REVISAR").any():
    raise ValueError("El diccionario de afiliaciones todavía contiene casos REVISAR.")

# Comprobar que la auditoría corresponde exactamente a la entrada
campos = {
    "Base_origen": "Base_origen",
    "Fuente_origen": "Fuente_origen",
    "indice": "indice",
    "Titulo": "Titulo",
    "Doi": "Doi",
    "Autor_norm": "Autor_original",
    "Afiliacion1": "Afiliacion1_original",
    "Afiliacion2": "Afiliacion2_original"
}

for campo_entrada, campo_auditoria in campos.items():
    iguales = entrada[campo_entrada].reset_index(drop=True).equals(
        auditoria[campo_auditoria].reset_index(drop=True)
    )
    if not iguales:
        raise ValueError(
            f"La auditoría no corresponde a la entrada en {campo_entrada}."
        )

print("Validaciones previas: OK")


In [ ]:
# Aplicar únicamente las tres columnas normalizadas
salida = entrada.copy()

salida["Autor_norm"] = auditoria["Autor_final"].values
salida["Afiliacion1"] = auditoria["Afiliacion1_final"].values
salida["Afiliacion2"] = auditoria["Afiliacion2_final"].values

# Validar que ninguna otra columna cambió
columnas_inmutables = [
    c for c in columnas
    if c not in ["Autor_norm", "Afiliacion1", "Afiliacion2"]
]

for c in columnas_inmutables:
    if not entrada[c].equals(salida[c]):
        raise ValueError(f"Se modificó una columna no permitida: {c}")

if len(salida) != len(entrada):
    raise ValueError("Cambió el número de filas.")

if salida["Autor_norm"].str.strip().eq("").any():
    raise ValueError("Hay autores vacíos después de la normalización.")

if salida["Afiliacion1"].str.strip().eq("").any():
    raise ValueError("Hay afiliaciones principales vacías después de la normalización.")

duplicadas_afiliacion = (
    salida["Afiliacion2"].str.strip().ne("")
    & salida["Afiliacion1"].eq(salida["Afiliacion2"])
)

if duplicadas_afiliacion.any():
    raise ValueError("Hay filas con Afiliacion1 y Afiliacion2 idénticas.")

print("Validaciones finales: OK")


In [ ]:
# Guardar resultado final de esta fase
salida.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:", archivo_salida)
print("Filas:", len(salida))
print("Columnas:", len(salida.columns))
print("Autores únicos finales:", salida["Autor_norm"].nunique())
print("Afiliaciones1 únicas finales:", salida["Afiliacion1"].nunique())
print("Afiliaciones2 no vacías:", salida["Afiliacion2"].str.strip().ne("").sum())
